# X-ray emission example

This example computes the absorbed spectrum for three different mock neutron stars with different X-ray luminosities and magnetic field values.

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import pathlib
import pickle
import scipy.integrate as integrate
import scipy.special as scsp

from scipy.integrate import trapz, quad
import mlpoppyns.simulator.basics.constants as const
import mlpoppyns.simulator.multiband_emission.emission_xray as xem
import mlpoppyns.simulator.interstellar_medium.nh_model as nhm
import mlpoppyns.simulator.interstellar_medium.xray_abs_cross_section as xabs
import utilities.plot_settings
from mlpoppyns.simulator.config_simulator import cfg

In [ ]:
# Load the interpolator function to evaluate the x-ray luminosity.
base_path = pathlib.Path("../../")
interpolator_Lx_path = base_path.joinpath(cfg["magneto-thermal_path"], "interpolator_Lx.pkl")

with open(interpolator_Lx_path, "rb") as f:
    Lx_interpolator = pickle.load(f)

In [ ]:
# Define some values for the age, initial magnetic field and sky position of the stars.
age = np.array([1.e4, 1.e4, 1.e4])
B = np.array([5.e12, 5.e13, 5.e14])
RA = np.array([60.0, 120.0, 250.0])
DEC = np.array([35.0, 45.0, 55.0])
d = np.array([5.0, 10.0, 15.0])
d_cm = d * const.KPC_TO_CM

# Apply the interpolator to find the corresponding X-ray luminosity.
Lx = Lx_interpolator.ev(age, B)

# Compute the observed black-body surface temperature and radius.
T_obs = xem.T_from_Lx(Lx)
R_obs = cfg["NS_radius"] / xem.gr_correction

# Evaluate the corresponding optical depth and thermal velocity of the magnetospheric plasma.
tau_res = xem.resonant_optical_depth(B)
tau_0 = tau_res / 2.0
beta_T = xem.beta_plasma(B)

# Define an array of energies and convert it from [eV] to [erg].
E = np.logspace(1.0, np.log10(20000), 1000)
E_erg = E * const.EV_TO_ERG

In [ ]:
# Compute the black-body intensity spectrum and convert it into [photon count cm^-2 s^-1 erg^-1 sterad^-1].
I_bb = xem.blackbody_intensity_spectrum(E_erg, T_obs)
I_ph_bb = I_bb / (E_erg)

# Compute the RCS spectrum and convert it into [erg cm^-2 s^-1 erg^-1 sterad^-1].
I_rcs = (
    xem.resonant_cyclotron_scat_spectrum(
        E_erg, E_erg, tau_0, beta_T, I_ph_bb, n_reflections=4
    )
    * E_erg
)

# Convert the intensities from [erg cm^-2 s^-1 erg^-1 sterad^-1] to [erg cm^-2 s^-1 eV^-1 sterad^-1].
I_bb = I_bb * const.EV_TO_ERG
I_rcs = I_rcs * const.EV_TO_ERG

# Estimate the N_H column density.
N_H = nhm.compute_NH(RA, DEC, d)
# Reshape N_H to make it compatible for broadcasting.
N_H = N_H[:, np.newaxis]

# Estimate the X-ray absorption cross section.
sigma_ISM = xabs.absorption_cross_section_tot(E, cfg["ISM_abundances"])

# Compute the absorbed intensity spectrum.
absorb_factor = np.exp(-sigma_ISM * N_H)
I_absorbed = absorb_factor * I_rcs

In [ ]:
# Convert intensities into fluxes as observed on Earth.
d_cm = d_cm[:, np.newaxis]
flux_bb = (R_obs / d_cm) ** 2 * np.pi * I_bb
flux_rcs = (R_obs / d_cm) ** 2 * np.pi * I_rcs
flux_absorbed = (R_obs / d_cm) ** 2 * np.pi * I_absorbed

In [ ]:
# Index to select a specific neutron star, choose between 0 and 2.
index = 2

fig, ax = plt.subplots(figsize=(12, 10))

# ax.set_xscale('log')
# ax.set_yscale('log')
ax.set_xlim(0.0, 2.0)
#ax.set_ylim(1.e-40,1.e-15)
ax.set_xlabel(r"Energy [keV]")
ax.set_ylabel(r"$S_{\rm X}(E)$ [erg cm$^{-2}$ s$^{-1}$ eV$^{-1}$]")

ax.plot(
    E * 1.0e-3,
    flux_bb[index, :],
    linestyle="--",
    linewidth=4,
    color="black",
    rasterized=True,
    label=r"BB intensity",
)
ax.plot(
    E * 1.0e-3,
    flux_rcs[index, :],
    linestyle="-",
    linewidth=4,
    color="tab:orange",
    alpha=1,
    rasterized=True,
    label=r"RCS spectrum",
)
ax.plot(
    E * 1.0e-3,
    flux_absorbed[index, :],
    linestyle="-",
    linewidth=4,
    color="tab:red",
    alpha=1,
    rasterized=True,
    label=r"Absorbed RCS spectrum",
)
plt.legend(frameon=False, loc=0)
plt.grid()

In [ ]:
# Compute the total absorbed flux in the energy band [0.1, 10] keV by integrating the absorbed RCS spectrum in energy.
flux_bb_abs_bolometric, flux_rcs_abs_bolometric, N_H = xem.flux_xray_absorbed(Lx, B, RA, DEC, d)
print(flux_bb_abs_bolometric)
print(flux_rcs_abs_bolometric)
print(N_H)

We compare the absorbed flux coming from a pure black-body with the absorbed flux of the black-body modified by the RCS process.
We use a simulation where we set a lower-limit threshold flux of $10^{-15}$ erg s$^{-1}$ cm$^{-2}$.

In [ ]:
path_to_simulation = pathlib.Path("../../data/example_simulation_magrot_det/")
df_x_sim = pd.read_pickle(
    pathlib.Path().joinpath(
        path_to_simulation, "survey_xray_flux_threshold_results.pkl.gz"
    ),
    compression="gzip",
)
df_x_sim.columns

B_x_sim = df_x_sim["B"]["[G]"].to_numpy()
S_x_rcs_abs_sim = df_x_sim["S_x_rcs_abs"]["[erg s^-1 cm^-2]"].to_numpy()
S_x_bb_abs_sim = df_x_sim["S_x_bb_abs"]["[erg s^-1 cm^-2]"].to_numpy()
NH_x_sim = df_x_sim["N_H"]["[cm^-2]"].to_numpy()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.set_xlabel(r"$S_{X, \rm{BB, abs}}$ [erg s$^{-1}$ cm$^{-2}$]")
ax.set_ylabel(r"$S_{X, \rm{RCS, abs}} / S_{X, \rm{BB, abs}}$")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(1e-19, 1.e-10)
ax.set_ylim(7e-1, 1e3)

sc = ax.scatter(
    S_x_bb_abs_sim[S_x_rcs_abs_sim > 1e-15],
    S_x_rcs_abs_sim[S_x_rcs_abs_sim > 1e-15] / S_x_bb_abs_sim[S_x_rcs_abs_sim > 1e-15],
    c=np.log10(NH_x_sim[S_x_rcs_abs_sim > 1e-15]),  # Color by log10 of B
    cmap="viridis",  # Choose a colormap
    edgecolor=None,
    s=60,  # Marker size
    alpha=1,
    rasterized=True,
)

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label(r"log$_{10}(N_{H} \, [{\rm cm}^{-2}])$")  # Label for the colorbar

#plt.legend(frameon=True, loc=0, fontsize=20)
plt.grid()

In this plot we represent the ratio between the absorbed RCS flux and the absorbed pure black-body flux for an example simulated population.
The colorcode represents the value of the $N_{H}$ of each simulated star.
Without the RCS process all sources below $S_{X, \rm{BB, abs}} = 10^{-15}$ erg s$^{-1}$ cm$^{-2}$ would be undetectable. Moreover the sources that show low black-body fluxes are the ones that are absorbed the most and without the RCS will be under the detection threshold. Fot those sources the ratio between the RCS flux and the pure black-body is larger due to the fact that the black-body contribution would be almost completely absorbed. 